In [0]:
MY_ID = "aasthathakur"   # change this — lowercase, no spaces
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS workspace;
CREATE SCHEMA IF NOT EXISTS workspace.capstone_yourname;   -- match MY_ID here
CREATE VOLUME IF NOT EXISTS workspace.capstone_yourname.raw;

In [0]:
import pyspark.sql.functions as F
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.capstone_{MY_ID}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.capstone_{MY_ID}.raw")
def W(d, n): d.write.mode('overwrite').option('header', True).csv(f'{VOL}/raw/{n}')

S = "if(hour between 18 and 20, 1.8, if(hour between 14 and 16, 0.6, 1.0))"
K = "if(dayofweek(trade_date) in (1, 7), 1.6, 1.0)"
C = "if(dayofweek(trade_date) in (1,7), .85, 1) * if(hour between 14 and 16, .45, 1)"

st = spark.range(12).selectExpr(
    "int(id) as s", "format_string('ST%02d', id + 1) as store_id",
    "format_string('CITY%d', id%6) as city",
    "if(id % 2 = 0, 'Mall', 'High Street') as format",
    "round(0.08 + rand(5) * 0.20, 3) as conv_base")
W(st.select('store_id', 'city', 'format'), 'stores')

gr = (st.crossJoin(spark.range(90).withColumnRenamed('id', 'd'))
        .crossJoin(spark.range(10, 22).withColumnRenamed('id', 'hour'))
        .selectExpr("*", "date_add(date'2025-04-01', int(d)) as trade_date"))
gr = gr.selectExpr("*", "(s * 90 + int(d)) * 12 + int(hour) - 10 as rn",
    f"int(round(30 * ({S}) * ({K}) * (0.7 + rand(6) * 0.6))) as fc")
gr = gr.selectExpr("*", f"int(round(fc * conv_base * ({C}))) as nb")

OFF = "pmod(rn, 54) = 0"
ERR = "(pmod(rn, 54) = 1 or pmod(rn, 216) = 2)"

W(gr.selectExpr("store_id", "trade_date", "hour",
    f"if({OFF}, 'OFFLINE', 'OK') as counter_status",
    f"case when {ERR} then 'ERR' when {OFF} then '0' else string(fc) end as footfall"),
  'footfall')

bl = gr.filter('nb > 0').withColumn('m', F.explode(F.sequence(F.lit(1), F.col('nb'))))
bl = bl.selectExpr(
    "format_string('B%02d%03d%02d%02d', s, int(d), int(hour), m) as bill_id",
    "store_id", "s * 90 + int(d) as k", "m", "hour",
    "format_string('%s %02d:%02d:00', string(trade_date), int(hour), m % 60) as bill_ts",
    "int(1 + rand(7) * 8) as items", "round(exp(6.0 + randn(8) * 0.7), 2) as bill_amount")

B = ['bill_id', 'store_id', 'bill_ts', 'items', 'bill_amount']
dup = bl.filter("hour = 12 and m = 1 and pmod(k, 8) = 3 and k < 1040")
ex = spark.range(135).selectExpr(
    "format_string('B9%06d', id) as bill_id",
    "if(id < 75, 'ST13', format_string('ST%02d', id % 12 + 1)) as store_id",
    "format_string('2025-05-%02d %02d:15:00', int(id) % 28 + 1, if(id < 75, 11 + int(id) % 9, 3 + int(id) % 3)) as bill_ts",
    "int(1 + pmod(id, 8)) as items", "round(400 + pmod(id, 900), 2) as bill_amount")

W(bl.select(*B).unionByName(dup.select(*B)).unionByName(ex.select(*B)), 'bills')
print("Done generating: stores, footfall, bills")

Done generating: stores, footfall, bills


In [0]:
from pyspark.sql.functions import current_timestamp, sha2, concat_ws, col

def bronze_load(name):
    raw = spark.read.option("header", True).csv(f"{VOL}/raw/{name}")
    df = raw.select([col(c).cast("string") for c in raw.columns] + [col("_metadata.file_path").alias("_source_file")])
    df = (df.withColumn("_ingested_at", current_timestamp())
            .withColumn("_row_hash", sha2(concat_ws("||", *df.columns), 256)))
    df.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.bronze_{name}")
    print(name, "bronze rows:", df.count())

bronze_load("stores")
bronze_load("footfall")
bronze_load("bills")

stores bronze rows: 12
footfall bronze rows: 12960
bills bronze rows: 86859


In [0]:
footfall_silver = spark.sql(f"""
SELECT store_id, try_cast(trade_date AS DATE) AS trade_date, int(hour) AS hour,
  counter_status,
  CASE WHEN counter_status = 'OFFLINE' THEN NULL
       ELSE try_cast(footfall AS INT) END AS footfall
FROM workspace.capstone_{MY_ID}.bronze_footfall
""")
footfall_silver.createOrReplaceTempView("footfall_silver")

In [0]:
bills_bronze = spark.table(f"workspace.capstone_{MY_ID}.bronze_bills")
bills_typed = bills_bronze.selectExpr(
    "bill_id", "store_id",
    "try_cast(bill_ts AS TIMESTAMP) AS bill_ts",
    "try_cast(items AS INT) AS items",
    "try_cast(bill_amount AS DOUBLE) AS bill_amount")

bills_deduped = bills_typed.dropDuplicates(["bill_id"])
stores_list = [r.store_id for r in spark.table(f"workspace.capstone_{MY_ID}.bronze_stores").select("store_id").collect()]

bills_rejects = bills_deduped.withColumn("reject_reason",
    F.when(~F.col("store_id").isin(stores_list), "unknown_store")
     .when((F.hour("bill_ts") < 10) | (F.hour("bill_ts") >= 22), "outside_trading_hours")
     .otherwise(None))

bills_silver = bills_rejects.filter(F.col("reject_reason").isNull()).drop("reject_reason")
bills_rejects.filter(F.col("reject_reason").isNotNull()).write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.silver_bill_rejects")
bills_silver.createOrReplaceTempView("bills_silver")

print("bills deduped:", bills_deduped.count())
print("bills silver (clean):", bills_silver.count())
print("bills rejected:", bills_rejects.filter(F.col("reject_reason").isNotNull()).count())

bills deduped: 86729
bills silver (clean): 86594
bills rejected: 135


In [0]:
silver_joined = spark.sql("""
SELECT f.store_id, f.trade_date, f.hour, f.counter_status, f.footfall,
  COALESCE(b.bills, 0) AS bills, COALESCE(b.revenue, 0.0) AS revenue,
  CASE WHEN f.footfall IS NOT NULL AND f.footfall > 0
       THEN COALESCE(b.bills,0) / f.footfall END AS conversion_rate,
  (f.footfall IS NOT NULL) AS sensor_ok
FROM footfall_silver f
LEFT JOIN (
  SELECT store_id, date(bill_ts) AS trade_date, hour(bill_ts) AS hour,
         count(*) AS bills, sum(bill_amount) AS revenue
  FROM bills_silver GROUP BY 1,2,3
) b ON f.store_id=b.store_id AND f.trade_date=b.trade_date AND f.hour=b.hour
""")
silver_joined.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.silver_store_hour")

In [0]:
gold = spark.sql(f"""
SELECT s.store_id, st.city, st.format, s.trade_date, s.hour,
  (dayofweek(s.trade_date) IN (1,7)) AS is_weekend,
  s.footfall, s.bills, s.revenue, s.conversion_rate, s.sensor_ok
FROM workspace.capstone_{MY_ID}.silver_store_hour s
JOIN workspace.capstone_{MY_ID}.bronze_stores st ON s.store_id = st.store_id
""")
gold.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.gold_store_hour")
print("Gold rows:", gold.count())

Gold rows: 12960


In [0]:
(spark.table(f"workspace.capstone_{MY_ID}.gold_store_hour")
      .coalesce(1).write.mode("overwrite").option("header", True)
      .csv(f"{VOL}/export/gold_store_hour"))
display(dbutils.fs.ls(f"{VOL}/export/gold_store_hour"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/capstone_aasthathakur/raw/export/gold_store_hour/_committed_1592116448626418947,_committed_1592116448626418947,201,1790283492000
dbfs:/Volumes/workspace/capstone_aasthathakur/raw/export/gold_store_hour/_committed_2039338602323417898,_committed_2039338602323417898,212,1790249655000
dbfs:/Volumes/workspace/capstone_aasthathakur/raw/export/gold_store_hour/_committed_5091703127515613163,_committed_5091703127515613163,201,1790249724000
dbfs:/Volumes/workspace/capstone_aasthathakur/raw/export/gold_store_hour/_started_1592116448626418947,_started_1592116448626418947,0,1790283492000
dbfs:/Volumes/workspace/capstone_aasthathakur/raw/export/gold_store_hour/part-00000-tid-1592116448626418947-73453ce6-cd0f-45e4-9f8b-3367130e4d20-502-1-c000.csv,part-00000-tid-1592116448626418947-73453ce6-cd0f-45e4-9f8b-3367130e4d20-502-1-c000.csv,1004918,1790283492000


In [0]:
display(spark.table(f"workspace.capstone_{MY_ID}.gold_store_hour").limit(15))

store_id,city,format,trade_date,hour,is_weekend,footfall,bills,revenue,conversion_rate,sensor_ok
ST01,CITY0,Mall,2025-04-01,10,false,null,6,4821.0,null,false
ST02,CITY1,High Street,2025-04-02,11,false,38,9,4934.25,0.23684210526315788,true
ST03,CITY2,Mall,2025-04-03,12,false,25,7,3528.7999999999997,0.28,true
ST04,CITY3,High Street,2025-04-04,13,false,27,7,3070.4300000000003,0.25925925925925924,true
ST05,CITY4,Mall,2025-04-05,14,true,24,1,182.05,0.041666666666666664,true
ST06,CITY5,High Street,2025-04-06,15,true,24,2,1312.68,0.08333333333333333,true
ST07,CITY0,Mall,2025-04-07,16,false,19,1,468.31,0.05263157894736842,true
ST08,CITY1,High Street,2025-04-08,17,false,24,6,3910.34,0.25,true
ST09,CITY2,Mall,2025-04-09,18,false,51,5,2607.84,0.09803921568627451,true
ST10,CITY3,High Street,2025-04-10,19,false,48,12,5424.86,0.25,true


In [0]:
gold = spark.table(f"workspace.capstone_{MY_ID}.gold_store_hour")

print("Total Gold rows:", gold.count())                                  # expect 12,960
print("sensor_ok = false count:", gold.filter("sensor_ok = false").count())  # expect 540

print("Bronze bill count:", spark.table(f"workspace.capstone_{MY_ID}.bronze_bills").count())
print("Silver revenue total:", gold.selectExpr("sum(revenue) as r").collect()[0]["r"])
print("Gold bills total:", gold.selectExpr("sum(bills) as b").collect()[0]["b"])   # expect 86,594

Total Gold rows: 12960
sensor_ok = false count: 540
Bronze bill count: 86859
Silver revenue total: 44766657.039999925
Gold bills total: 86594


In [0]:
gold_rev = spark.table(f"workspace.capstone_{MY_ID}.gold_store_hour").selectExpr("sum(revenue) as r").collect()[0]["r"]
silver_rev = bills_silver.selectExpr("sum(bill_amount) as r").collect()[0]["r"]
print("Gold vs Silver revenue difference:", gold_rev - silver_rev)

Gold vs Silver revenue difference: -4.544854164123535e-07
